# Day 6 — PySpark Practice Questions: groupBy, rollup, cube

**3 questions** · Easy → Medium  
Solutions are in hidden cells below each question — try first!

In [ ]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day6_PySpark_Practice') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

sales_data = [
    (1,  'North', 'Electronics', 'Laptop',   '2024-01-05', 1200.0),
    (2,  'North', 'Electronics', 'Phone',    '2024-01-10',  800.0),
    (3,  'North', 'Furniture',   'Chair',    '2024-01-12',  300.0),
    (4,  'North', 'Furniture',   'Desk',     '2024-01-15',  600.0),
    (5,  'South', 'Electronics', 'Tablet',   '2024-01-08',  500.0),
    (6,  'South', 'Electronics', 'Laptop',   '2024-01-20', 1100.0),
    (7,  'South', 'Furniture',   'Sofa',     '2024-01-18',  900.0),
    (8,  'South', 'Furniture',   'Wardrobe', '2024-01-22', 1500.0),
    (9,  'East',  'Electronics', 'Phone',    '2024-01-09',  750.0),
    (10, 'East',  'Furniture',   'Chair',    '2024-01-14',  280.0),
]
sales_schema = StructType([
    StructField('sale_id',   IntegerType(), False),
    StructField('region',    StringType(),  True),
    StructField('category',  StringType(),  True),
    StructField('product',   StringType(),  True),
    StructField('sale_date', StringType(),  True),
    StructField('revenue',   DoubleType(),  True),
])
df_sales = spark.createDataFrame(sales_data, schema=sales_schema)
print('Rows:', df_sales.count())
df_sales.show()

---
## Q1 (Easy) — Total Revenue by Region and Category

Using `.groupBy().agg()`, compute:
- `num_sales` — count of sale records
- `total_revenue` — sum of revenue (rounded to 2 dp)

Grouped by `region` and `category`, ordered by `region`, `category`.

**Expected:** 6 rows

**Warm-ups:**
- What's the difference between `F.count('*')` and `F.count('revenue')` on this data?
- How do you pass multiple aggregations to `.agg()`?
- What alias gives you a column named `total_revenue`?

In [ ]:
df_q1 = None
# YOUR CODE HERE

In [ ]:
# --- assert ---
assert df_q1 is not None, 'df_q1 not set'
assert df_q1.count() == 6, f'Expected 6 rows, got {df_q1.count()}'

cols = set(df_q1.columns)
assert 'region' in cols and 'category' in cols and 'total_revenue' in cols, \
    f'Missing expected columns. Got: {cols}'

# North Electronics: 1200 + 800 = 2000
north_elec = df_q1.filter((F.col('region') == 'North') & (F.col('category') == 'Electronics')).collect()[0]
assert abs(north_elec['total_revenue'] - 2000.0) < 0.01, f'North Electronics total wrong: {north_elec["total_revenue"]}'

print('Q1 passed!')
df_q1.show()

### Solution

In [ ]:
# SOLUTION
df_q1 = (
    df_sales
    .groupBy('region', 'category')
    .agg(
        F.count('*').alias('num_sales'),
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
    )
    .orderBy('region', 'category')
)
df_q1.show()

---
## Q2 (Medium) — ROLLUP with Labeled Subtotals

Use `.rollup('region', 'category')` to produce:
- Detail rows (region + category)
- Region subtotals
- Grand total

Replace NULL values: `region=NULL` → `'ALL REGIONS'`, `category=NULL` → `'ALL CATEGORIES'`.

Store result in `df_q2`.

**Expected row count:** 10 (6 detail + 3 region subtotals + 1 grand total)

**Warm-ups:**
- What does NULL mean in rollup output? Does it mean the data is missing?
- Which function replaces NULL with a literal string?
- Why is `.withColumn()` used AFTER `.agg()` and not inside it?

In [ ]:
df_q2 = None
# YOUR CODE HERE
# Use .rollup() and F.coalesce() to label NULLs

In [ ]:
# --- assert ---
assert df_q2 is not None, 'df_q2 not set'
assert df_q2.count() == 10, f'Expected 10 rows, got {df_q2.count()}'

# Grand total row should exist
grand = df_q2.filter(
    (F.col('region') == 'ALL REGIONS') & (F.col('category') == 'ALL CATEGORIES')
).collect()
assert len(grand) == 1, 'Grand total row not found or wrong label'
assert abs(grand[0]['total_revenue'] - 7930.0) < 0.01, f'Grand total wrong: {grand[0]["total_revenue"]}'

# North subtotal: 1200+800+300+600 = 2900
north_sub = df_q2.filter(
    (F.col('region') == 'North') & (F.col('category') == 'ALL CATEGORIES')
).collect()
assert len(north_sub) == 1, 'North subtotal row not found'
assert abs(north_sub[0]['total_revenue'] - 2900.0) < 0.01, f'North subtotal wrong: {north_sub[0]["total_revenue"]}'

print('Q2 passed!')
df_q2.show()

### Solution

In [ ]:
# SOLUTION
df_q2 = (
    df_sales
    .rollup('region', 'category')
    .agg(F.round(F.sum('revenue'), 2).alias('total_revenue'))
    .withColumn('region',   F.coalesce(F.col('region'),   F.lit('ALL REGIONS')))
    .withColumn('category', F.coalesce(F.col('category'), F.lit('ALL CATEGORIES')))
    .orderBy('region', 'category')
)
df_q2.show()

---
## Q3 (Medium) — CUBE with Level Labels

Use `.cube('region', 'category')` to produce all grouping combinations. Add a `grouping_level` column that labels each row:
- `'region+category'` — detail rows
- `'region only'` — region subtotals
- `'category only'` — category subtotals
- `'grand total'` — the one grand total row

Use `F.grouping('col')` to detect the level.

Store result in `df_q3`.

**Expected row count:** 12 (6 + 3 + 2 + 1)

**Warm-ups:**
- What does `F.grouping('region')` return for a category-only subtotal row?
- How does CUBE differ from ROLLUP — which levels does it add?
- What is the total revenue for the 'Electronics' category-only row?

In [ ]:
df_q3 = None
# YOUR CODE HERE
# Use .cube() + F.grouping() + F.when() to add grouping_level column

In [ ]:
# --- assert ---
assert df_q3 is not None, 'df_q3 not set'
assert df_q3.count() == 12, f'Expected 12 rows, got {df_q3.count()}'

assert 'grouping_level' in df_q3.columns, 'grouping_level column missing'

# Check all 4 level labels exist
levels = {r['grouping_level'] for r in df_q3.select('grouping_level').distinct().collect()}
assert levels == {'region+category', 'region only', 'category only', 'grand total'}, \
    f'Wrong level labels: {levels}'

# Electronics category total: 1200+800+500+1100+750 = 4350
elec_cat = df_q3.filter(
    (F.col('category') == 'Electronics') & (F.col('grouping_level') == 'category only')
).collect()
assert len(elec_cat) == 1, 'Electronics category-only row not found'
assert abs(elec_cat[0]['total_revenue'] - 4350.0) < 0.01, f'Electronics total wrong: {elec_cat[0]["total_revenue"]}'

print('Q3 passed!')
df_q3.orderBy('grouping_level', 'region', 'category').show(20)

### Solution

In [ ]:
# SOLUTION
df_q3 = (
    df_sales
    .cube('region', 'category')
    .agg(
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.grouping('region').alias('region_rolled'),
        F.grouping('category').alias('cat_rolled'),
    )
    .withColumn('grouping_level',
        F.when((F.col('region_rolled') == 0) & (F.col('cat_rolled') == 0), 'region+category')
         .when((F.col('region_rolled') == 0) & (F.col('cat_rolled') == 1), 'region only')
         .when((F.col('region_rolled') == 1) & (F.col('cat_rolled') == 0), 'category only')
         .otherwise('grand total')
    )
    .withColumn('region',   F.coalesce(F.col('region'),   F.lit('ALL')))
    .withColumn('category', F.coalesce(F.col('category'), F.lit('ALL')))
    .drop('region_rolled', 'cat_rolled')
    .orderBy('grouping_level', 'region', 'category')
)
df_q3.show(20)

In [ ]:
spark.stop()
print('Spark stopped.')